# B2 - Person B, GPU runs for the compact residual U-Net (denoise-study)

**Kaggle session settings:** Accelerator **one GPU** (single T4 *or* P100 - **not** "T4 x2", the code uses one GPU) |
Internet **ON** only for the clone (or use an attached code dataset) | Inputs: `bsd-data` (pinned), `denoise-shared`
(pinned, from A), and your previous `denoise-B-results` version **if you are resuming**.

**One focused job list per session** (set `RUN` in the config cell): `tests` -> `bench` -> `pilot` -> `train_*` -> `infer_*`.
No debugging on the GPU quota: if something breaks, fix it on CPU (B1 / laptop), push, re-run.
Never run two GPU notebooks at once (same quota drains twice as fast). Turn the accelerator off when finished.

In [ ]:
# ---- CONFIG: edit this cell only ----
NOTEBOOK_NAME = "B2_gpu_unet"
REPO_URL = "https://github.com/abhinavpareek655/denoise-study.git"  # repo URL; private repo -> use Kaggle Secrets, never paste a token here.
BRANCH = "main"
PINNED_COMMIT = "db58a0c"            # set for every run whose results you keep; P and G of one seed should share it

USE_SHARED_FOR_PIPELINE = False   # see B1; check the printed common.paths first
REQUIRE_GPU = True

# Pick jobs for THIS session. Valid names are the keys of PRESETS below.
#   First GPU session : ["tests", "bench"]
#   After bench + A   : ["pilot"]                       (needs the shared step budget in configs/train/train.yaml)
#   After LAMBDA_FROZEN is set and pushed: ["train_P_s0"], then ["train_G_s0"], ["train_P_s1"], ["train_G_s1"]
#   Inference         : ["infer_P_s0", ...]
RUN = ["tests", "bench"]

SESSION_LIMIT_H = 9.0         # plan for <= 9 h (Section 24); the real Kaggle limit is up to 12 h
MAX_HOURS_PER_RUN = 8.5       # passed as --max_hours to full training runs (clean checkpoint + exit)

In [ ]:
# ---- Bootstrap (identical in B1 and B2): helpers, repo clone/pin, dataset lookup, env vars ----
import os, sys, json, time, shutil, hashlib, subprocess, datetime, importlib
from pathlib import Path

WORK = Path("/kaggle/working")          # the ONLY directory Kaggle keeps as notebook output
REPO = WORK / "denoise-study"
LOG_DIR = WORK / "notebook_logs"
LOG_DIR.mkdir(exist_ok=True)
assert Path("/kaggle").exists(), "This notebook is written for Kaggle (paths come from /kaggle)."


def sh(cmd, cwd=None, check=True, log=None):
    """Run a shell command, stream output live, optionally tee it to a log file. Returns exit code."""
    print(f"$ {cmd}", flush=True)
    t0 = time.time()
    run_dir = cwd or (REPO if REPO.exists() else None)
    p = subprocess.Popen(cmd, shell=True, cwd=run_dir, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    lf = open(log, "a") if log else None
    for line in p.stdout:
        print(line, end="", flush=True)
        if lf:
            lf.write(line)
            lf.flush()
    rc = p.wait()
    if lf:
        lf.close()
    print(f"[exit {rc} after {time.time() - t0:.0f}s]", flush=True)
    if check and rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")
    return rc


def find_dir_containing(name, root="/kaggle/input", max_depth=3):
    """Breadth-first search under `root` for a directory that contains `name`."""
    queue = [(Path(root), 0)]
    while queue:
        d, depth = queue.pop(0)
        if not d.is_dir():
            continue
        if (d / name).exists():
            return d
        if depth < max_depth:
            queue.extend((c, depth + 1) for c in sorted(d.iterdir()) if c.is_dir())
    return None


# --- code: clone from GitHub (Internet ON) or copy from an attached code dataset (Internet OFF) ---
if not REPO.exists():
    if REPO_URL:
        assert "<" not in REPO_URL, "Edit REPO_URL in the config cell first."
        sh(f"git clone --branch {BRANCH} {REPO_URL} {REPO}", cwd=WORK)
    else:
        src = find_dir_containing("PROJECT_CONTEXT.md")
        assert src is not None, "No REPO_URL and no attached code dataset containing PROJECT_CONTEXT.md"
        shutil.copytree(src, REPO)
if (REPO / ".git").exists() and REPO_URL:
    sh("git fetch --all --prune")
    sh(f"git checkout {PINNED_COMMIT}" if PINNED_COMMIT else f"git checkout {BRANCH} && git pull --ff-only")
COMMIT = subprocess.run("git rev-parse HEAD", shell=True, cwd=REPO, capture_output=True,
                        text=True).stdout.strip() or "no-git"
print("code commit:", COMMIT)

# --- data: the attached bsd-data dataset must contain BSD68/ and BSD400/ ---
DATA = find_dir_containing("BSD68")
assert DATA is not None and (DATA / "BSD400").exists(), "Attach the bsd-data dataset (BSD68/ and BSD400/)."
os.environ["DENOISE_DATA"] = str(DATA)
SHARED_DIR = find_dir_containing("manifest.json")      # A's denoise-shared dataset, if attached
if USE_SHARED_FOR_PIPELINE:
    assert SHARED_DIR is not None, "USE_SHARED_FOR_PIPELINE=True but no denoise-shared dataset is attached"
    os.environ["DENOISE_SHARED"] = str(SHARED_DIR)
print("DENOISE_DATA  =", os.environ["DENOISE_DATA"])
print("DENOISE_SHARED=", os.environ.get("DENOISE_SHARED", "(unset: repo artifacts/ is used)"))
print("shared dataset found at:", SHARED_DIR)

os.chdir(REPO)
sys.path.insert(0, str(REPO))
paths = importlib.import_module("common.paths")
print("\ncommon.paths resolves (CHECK these look right before running anything):")
for n in sorted(dir(paths)):
    v = getattr(paths, n)
    if not n.startswith("_") and isinstance(v, (str, Path, dict)):
        print(f"  {n} = {v}")
print("  functions:", [n for n in dir(paths) if not n.startswith("_") and callable(getattr(paths, n))])


In [ ]:
# ---- Environment capture (Section 24.3): saved to artifacts/logs/env_<timestamp>.json ----
import platform
import numpy, scipy, skimage, pandas


def _cap(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60)
    return r.stdout.strip()


env = {
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "notebook": NOTEBOOK_NAME,
    "code_commit": COMMIT,
    "python": platform.python_version(),
    "numpy": numpy.__version__, "scipy": scipy.__version__,
    "scikit_image": skimage.__version__, "pandas": pandas.__version__,
    "cpu_model": _cap("lscpu | grep 'Model name' | head -1 | cut -d: -f2"),
    "cpu_count": os.cpu_count(),
    "gpu": _cap("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader") or "none",
    "data_dir": str(DATA), "shared_dir": str(SHARED_DIR),
}
try:
    import torch
    env.update(torch=torch.__version__, cuda=torch.version.cuda, cuda_available=torch.cuda.is_available())
except Exception as e:  # torch missing is reported, not fatal, in the CPU notebook
    env["torch"] = f"unavailable: {e!r}"
(REPO / "artifacts" / "logs").mkdir(parents=True, exist_ok=True)
env_path = REPO / "artifacts" / "logs" / f"env_{datetime.datetime.now():%Y%m%d_%H%M%S}.json"
env_path.write_text(json.dumps(env, indent=2))
print(json.dumps(env, indent=2))

# The contract wants common/versions.py output here; its API is not in the notes, so list what exists:
try:
    vers = importlib.import_module("common.versions")
    print("common.versions public names:", [n for n in dir(vers) if not n.startswith("_")])
except Exception as e:
    print("common.versions failed to import:", repr(e))


In [ ]:
import torch
print("cuda available:", torch.cuda.is_available())
if REQUIRE_GPU:
    assert torch.cuda.is_available(), "No GPU: set Accelerator to a single T4/P100 (or set REQUIRE_GPU=False for CPU-only jobs)"
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print("GPU:", GPU_NAME, "| run P and G of the same seed on the same GPU type (Section 24.8)")

## Restore checkpoints from a previous session
`/kaggle/working` starts **empty** in every new session. To resume (or to run inference on trained models), attach your
previous `denoise-B-results` version as an input; this cell copies `artifacts/checkpoints/<model_id>/` back so that
re-running the *same* `scripts.train` command resumes from `last.pt`. Attach **one** results version only.

In [ ]:
restored = []
for last in sorted(Path("/kaggle/input").rglob("last.pt")):
    mid = last.parent.name
    dst = REPO / "artifacts" / "checkpoints" / mid
    if not dst.exists():
        shutil.copytree(last.parent, dst)
        restored.append(mid)
print("restored checkpoints:", restored if restored else "none (fresh start)")
for d in sorted((REPO / "artifacts" / "checkpoints").glob("*")) if (REPO / "artifacts" / "checkpoints").exists() else []:
    print("  ", d.name, sorted(p.name for p in d.iterdir()))

## Job runner
Presets are plain `scripts.*` commands from the README stage table. Full training runs get `--max_hours` computed from the
time left in this session. The runner does **not** know whether a stopped run is finished: after each training job look at the
step in the log; if it has not reached the configured final step, run the same preset again in the next session.

In [ ]:
sh("python -m scripts.train --help | head -n 40", check=False)
sh("python -m scripts.infer --help | head -n 25", check=False)

from common import protocol as PR
SESSION_START = time.time()

PRESETS = {
    "tests": ["python -m pytest -q tests/test_torch_parts.py"],
    "bench": ["python -m scripts.train --arch unet --variant P --seed 0 --bench 200"],
    "pilot": [f"python -m scripts.train --arch unet --variant G --lam {lam} --pilot" for lam in PR.LAMBDA_GRID],
}
for v in PR.LOSS_VARIANTS:
    for s in PR.TRAIN_SEEDS:
        PRESETS[f"train_{v}_s{s}"] = [f"python -m scripts.train --arch unet --variant {v} --seed {s}"]
        PRESETS[f"infer_{v}_s{s}"] = [f"python -m scripts.infer --method unet --model-id unet_{v}_s{s}"]


def hours_left():
    return SESSION_LIMIT_H - (time.time() - SESSION_START) / 3600


def run_preset(name):
    if name.startswith("train_G"):
        pr = importlib.reload(importlib.import_module("common.protocol"))
        assert pr.LAMBDA_FROZEN is not None, (
            "LAMBDA_FROZEN is None: finish the pilot, set it ONCE in common/protocol.py (both networks), "
            "commit, push, then re-clone with the new commit.")
        print("LAMBDA_FROZEN =", pr.LAMBDA_FROZEN)
    for cmd in PRESETS[name]:
        if "scripts.train" in cmd and "--bench" not in cmd and "--pilot" not in cmd:
            budget = min(MAX_HOURS_PER_RUN, hours_left() - 0.25)
            if budget < 0.5:
                print(f"Only {hours_left():.2f} h left in this session: not starting {name}.")
                return
            cmd += f" --max_hours {budget:.2f}"
        sh(cmd, log=LOG_DIR / f"{name}.log")

In [ ]:
for name in RUN:
    assert name in PRESETS, f"unknown preset {name}; valid: {sorted(PRESETS)}"
for name in RUN:
    print(f"\n########## {name}   (session time left: {hours_left():.2f} h) ##########")
    run_preset(name)

## Review what the run produced
Prints `meta.json`, the tail of `train_log.csv`, and saves a validation curve per model to `artifacts/figures/`.
Sanity rules (Section 14.6): validation PSNR must beat the noisy input at all three sigmas and should clearly beat the best
classical result on validation images; if not, suspect a bug (scale, residual sign, normalization, too few steps).

In [ ]:
import re
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG = REPO / "artifacts" / "figures"
FIG.mkdir(parents=True, exist_ok=True)


def review_run(d):
    print(f"\n=== {d.name} ===  files: {sorted(p.name for p in d.iterdir())}")
    meta = d / "meta.json"
    hw = None
    if meta.exists():
        m = json.loads(meta.read_text())
        hw = m.get("hardware")
        print(json.dumps(m, indent=1)[:1500])
    else:
        print("meta.json not present yet")
    lg = d / "train_log.csv"
    if not lg.exists():
        return hw
    df = pd.read_csv(lg)
    print(df.tail(5).to_string(index=False))
    step = next((c for c in df.columns if "step" in c.lower()), None)
    vals = [c for c in df.columns if "val" in c.lower() and "psnr" in c.lower()]
    if step and vals:
        ax = df.plot(x=step, y=vals, figsize=(6, 3.5), marker="o", ms=3)
        ax.set_xlabel("training step")
        ax.set_ylabel("validation PSNR (dB)")
        ax.set_title(d.name)
        out = FIG / f"train_curve_{d.name}.png"
        ax.figure.savefig(out, dpi=200, bbox_inches="tight")
        plt.show()
        print("saved", out)
    return hw


ckdir = REPO / "artifacts" / "checkpoints"
hardware = {}
for d in sorted(ckdir.glob("unet_*")) if ckdir.exists() else []:
    hardware[d.name] = review_run(d)

for s in PR.TRAIN_SEEDS:
    a, b = hardware.get(f"unet_P_s{s}"), hardware.get(f"unet_G_s{s}")
    if a and b and a != b:
        print(f"WARNING seed {s}: P trained on '{a}' but G on '{b}' (Section 24.8: not bit-comparable)")

In [ ]:
def du_mb(p):
    return sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file()) / 1e6


for sub in ("checkpoints", "outputs", "logs", "figures"):
    p = REPO / "artifacts" / sub
    if p.exists():
        print(f"artifacts/{sub:11s} {du_mb(p):8.1f} MB")
print(f"/kaggle/working total: {du_mb(WORK):.0f} MB   (limit is about 20 GB)   session time used: {(time.time()-SESSION_START)/3600:.2f} h")

## After the run (not automatic)
1. **bench:** write down steps/second for U-Net; send it to A. Both networks must get the **same** step budget in
   `configs/train/train.yaml` - edit it in the repo on your laptop, push, pin the new commit. Do not edit it inside Kaggle.
2. **pilot:** record the validation criterion for lambda 0.1 / 0.5 / 1.0 in `docs/lambda_pilot.md`. `LAMBDA_FROZEN` is set
   **once, for both networks**, together with A (Section 14.4), then committed and logged in Section 22.
3. **train / infer:** **Save Version** (commit) so `/kaggle/working` is stored, then publish `artifacts/checkpoints`,
   `artifacts/outputs`, `artifacts/logs` as the next version of **`denoise-B-results`**; note the version number in
   Section 22. A session that ends without a saved version loses everything in `/kaggle/working`.
4. Fill `docs/model_provenance.md` (real parameter counts from B1, hardware, steps) from each `meta.json`.